# TCC - Análise Preditiva de Risco Acadêmico Escolar
## CRISP-DM: 1. Business Understanding

Este notebook apresenta a primeira fase do CRISP-DM aplicada ao projeto: o entendimento do negócio. Nesta fase, o foco é definir o problema institucional, os usuários, o valor pedagógico esperado, os alvos preditivos e os critérios de sucesso da solução, sem expor dados sensíveis nem detalhes privados do banco.



## Escopo desta série CRISP-DM

Esta série organiza a demonstração do TCC nas seis fases clássicas do CRISP-DM:

1. Business Understanding
2. Data Understanding
3. Data Preparation
4. Modeling
5. Evaluation
6. Deployment

Neste item 1, o objetivo é esclarecer o problema e definir o que será considerado sucesso. Nas próximas fases, a demonstração deve continuar usando apenas amostras anonimizadas, agregadas ou sintéticas, sem expor SQL real, credenciais ou a estrutura completa do banco institucional.




## Contexto do problema

Em uma instituição escolar, professores, coordenadores e secretaria acompanham informações de alunos em diferentes frentes: notas, faltas, situação de matrícula, turma, série, curso, pagamentos e vínculos com professores. O problema é que esses sinais normalmente são analisados de forma separada e reativa.

Na prática, muitas intervenções acontecem apenas depois que o aluno já apresenta queda relevante de desempenho, excesso de faltas ou risco de reprovação. O projeto busca antecipar esses sinais, transformando histórico escolar em relatórios que apoiem a tomada de decisão pedagógica.



## Problema de negócio

A escola precisa identificar, com antecedência, quais alunos podem apresentar piora acadêmica em uma disciplina ou entrar em uma faixa de risco. Essa identificação não deve depender apenas da última nota observada, porque o desempenho do aluno pode ser influenciado por histórico anterior, tendência recente, faltas acumuladas, diferença em relação a turma, quantidade de etapas restantes, contexto da série e outros fatores.

O desafio do projeto é transformar dados históricos em uma visão operacional simples: quem precisa de atenção, em qual disciplina, em qual turma, com qual nível de risco e com quais sinais justificando esse alerta. Em outras palavras, a escola não quer apenas constatar a reprovação; ela quer agir antes.



### Exemplo fixo do problema de negócio

<div style="font-size: 1em; overflow-x: auto;">

| papel | situação | decisão |
| --- | --- | --- |
| Professor | queda recente em Matemática | intervenção pedagógica antes da próxima etapa |
| Coordenação | turma com vários alunos em risco alto | priorizar acompanhamento por série e disciplina |
| Secretaria | faltas e pendências se acumulando | acionar família e apoiar regularização |

</div>



## Objetivo geral

Construir uma pipeline preditiva capaz de estimar a próxima nota do aluno e indicar risco acadêmico, gerando relatórios, rankings e visualizações operacionais para professores, coordenação e secretaria com foco em acompanhamento pedagógico preventivo.


## Objetivos específicos

- Extrair dados escolares tratados a partir de CSVs canônicos, sem expor as consultas SQL reais.
- Unificar notas, faltas, pagamentos e vínculos de professor em uma base temporal de modelagem.
- Criar atributos históricos por aluno, disciplina, etapa, série e ano letivo.
- Prever a próxima nota do aluno por regressão.
- Classificar se a próxima nota tende a ficar abaixo da linha de risco pedagógico.
- Comparar modelos de aprendizado de máquina com baselines simples, como última nota e médias históricas, usando métricas coerentes com cada tarefa.
- Gerar relatórios finais por perfil de uso: professor, coordenação e secretaria.
- Disponibilizar visualização operacional em dashboard Streamlit.


## Usuários e interessados

<div style="font-size: 1em; overflow-x: auto;">

| Perfil | Interesse principal | Exemplo de uso |
|---|---|---|
| Professor | Ver alunos com risco em suas disciplinas e turmas | Priorizar revisão, reforço ou acompanhamento individual |
| Coordenação | Identificar concentração de risco por série, turma, disciplina e professor | Planejar intervenções pedagógicas e reuniões de acompanhamento |
| Secretaria | Apoiar visão cadastral, histórica e operacional | Acompanhar situações que exigem contato, registro ou verificação |
| Gestão escolar | Entender padrões gerais de desempenho e risco | Avaliar a necessidade de ações institucionais |

</div>



## Perguntas de negócio

- Qual tende a ser a próxima nota de um aluno em determinada disciplina?
- O aluno está caminhando para uma nota abaixo da linha de risco acadêmico?
- Quais alunos devem receber atenção primeiro?
- Quais disciplinas, séries ou turmas concentram maior erro ou maior risco?
- Quais professores possuem maior quantidade de alunos em risco, considerando suas turmas e disciplinas?
- Qual é o mínimo de histórico necessário para gerar uma avaliação mais confiável?
- O modelo está realmente superando regras simples, como repetir a última nota como previsão?




## Tipo de problema de ciência de dados

O projeto possui dois objetivos técnicos complementares:

<div style="font-size: 1em; overflow-x: auto;">

| Objetivo | Tipo de tarefa | Alvo previsto | Exemplo |
|---|---|---|---|
| Previsão de nota | Regressão | `target_nota_proxima` | prever que a próxima média em Matemática será 5.8 |
| Alerta de risco | Classificação binária | `target_risco_proxima` | indicar risco quando a próxima nota esperada ou historicamente observada fica abaixo de 6.0 |

</div>

A regressão responde a pergunta 'qual nota provavelmente vem a seguir?'. A classificação responde a pergunta operacional 'o aluno precisa de atenção por risco acadêmico?'.




## Exemplo conceitual do alvo de previsão

Imagine um aluno com as seguintes notas em Língua Portuguesa:

<div style="font-size: 1em; overflow-x: auto;">

| Ano | Etapa | Nota observada | Próxima nota usada como alvo |
|---|---:|---:|---:|
| 2025 | 1 | 7.0 | 6.2 |
| 2025 | 2 | 6.2 | 5.5 |
| 2025 | 3 | 5.5 | 6.0 |

</div>

Na linha da etapa 2, por exemplo, o modelo enxerga o histórico disponível até aquele momento e tenta prever a etapa seguinte. Se a próxima nota real for 5.5, o alvo de regressão daquela linha é 5.5. O alvo de classificação será 1, porque 5.5 está abaixo de 6.0.




### Exemplo fixo do alvo de previsão

<div style="font-size: 1em; overflow-x: auto;">

| NomeSerie | NomeDisciplina | nota_atual | faltas_acumuladas | pagamentos_pendentes_ano | target_nota_proxima | target_risco_proxima |
| --- | --- | --- | --- | --- | --- | --- |
| 9º Ano | Matemática 2 | 4.0 | 8 | 2 | 3.5 | 1 |
| 1ª Série | Química 1 | 7.4 | 1 | 0 | 7.1 | 0 |

</div>



## Fontes de dados esperadas

O repositório público não depende diretamente do desenho completo do banco institucional. A interface pública da pipeline são CSVs canônicos em `artifacts/database/csv/`, conforme o contrato descrito em `docs/ENTRADA_DE_DADOS_E_CONTRATOS.md`:

- `aluno.csv`: vínculos escolares do aluno.
- `media_nota_aluno.csv`: notas e médias por disciplina e etapa.
- `faltas_aluno.csv`: eventos de falta por disciplina e etapa.
- `pagamento_aluno.csv`: informações financeiras agregáveis por aluno e ano.
- `responsaveis_aluno.csv`: vínculos de responsáveis, quando usados em relatórios.
- `professor_disciplina.csv`: vínculo entre professor, turma e disciplina quando existir.

As consultas SQL reais e a preparação física do banco ficam fora do Git, em camada privada local.



## Critérios de sucesso técnico

<div style="font-size: 1em; overflow-x: auto;">

| Parte | Métrica principal | Interpretação |
|---|---|---|
| Previsão de nota | MAE | erro médio absoluto entre nota prevista e nota real |
| Previsão de nota | RMSE | penaliza erros grandes com mais intensidade |
| Previsão de nota | acerto até 0.5 ponto | porcentagem de previsões com erro máximo de meio ponto |
| Alerta de risco | Precision | quando o modelo alerta risco, quantas vezes estava correto |
| Alerta de risco | Recall | entre os alunos que realmente entraram em risco, quantos foram encontrados |
| Alerta de risco | F1 | equilíbrio entre precision e recall |

</div>

Na implementação atual, a seleção principal usa MAE para regressão e F1 para classificação, enquanto RMSE, acerto até 0.5 ponto, precision e recall ajudam a interpretar o comportamento do modelo. No contexto escolar, um bom modelo não deve ser avaliado apenas por uma porcentagem geral: ele precisa errar pouco nas notas e também identificar bem alunos em risco, porque falso negativo pode significar deixar de apoiar um aluno que precisava de intervenção.



## Critérios de sucesso pedagógico

- O professor consegue entender quais alunos priorizar.
- A coordenação consegue enxergar concentrações de risco por turma, série, disciplina e professor.
- A secretaria consegue apoiar a leitura operacional sem depender de planilhas isoladas.
- O projeto não expõe dados pessoais nem consultas internas do banco.
- Os relatórios ajudam na ação preventiva, não apenas na constatação tardia de reprovação.



## Restrições e cuidados

- Dados escolares são sensíveis e devem permanecer anonimizados fora do ambiente autorizado.
- Alunos podem entrar e sair da escola, portanto nem todos terão histórico suficiente.
- Turmas e disciplinas podem existir sem professor vinculado; isso deve ser tratado como ausência legítima de dado, não como erro.
- A avaliação precisa respeitar a ordem temporal, treinando com anos anteriores e testando em anos posteriores.
- O modelo deve ser usado como apoio à decisão, não como decisão automática sobre o aluno.




## Resultado esperado ao final do projeto

Ao final da pipeline, espera-se gerar:

- datasets de modelagem em `artifacts/pipeline/`;
- relatórios técnicos de previsão e risco;
- análises de erro por disciplina, série e faixa de nota;
- relatórios finais para professor, coordenação e secretaria;
- ranking executivo de alunos prioritários e ranking de professores por volume de risco;
- dashboard Streamlit para visualização dos principais indicadores.


## Decisão desta fase

O projeto deve seguir para a fase de Data Understanding usando apenas dados já extraídos, tratados e anonimizados. A próxima etapa deve mostrar o formato dos CSVs, suas colunas, quantidade de registros, períodos disponíveis, exemplos agregados, chaves de integração e primeiros problemas encontrados, sem revelar dados pessoais nem SQL institucional.




## Saída didática da fase

Nesta fase ainda não existe transformação física dos dados nem treinamento de modelo. O que muda aqui é a definição do problema que guiará todas as etapas seguintes.

Quando este item termina, o projeto já deixou claro:
- qual pergunta institucional quer responder;
- quais dados precisará integrar depois;
- qual será o alvo da regressão e o alvo da classificação;
- quais métricas dirão se a solução ficou útil;
- quais perfis da escola receberão as entregas finais.

Ou seja, a principal saída do Business Understanding não é um arquivo de dados novo, mas um contrato de decisão: o que o projeto deve prever, para quem, com que critério de sucesso e com quais restrições de segurança.




### Exemplo fixo da saída desta fase

<div style="font-size: 1em; overflow-x: auto;">

| fase | saída | uso |
| --- | --- | --- |
| Business Understanding | problema definido | alinhar o que a escola quer prevenir |
| Data Understanding | mapa das bases | saber o que existe e o que falta |
| Deployment | relatórios e dashboard | transformar previsões em ação escolar |

</div>

